In [ ]:
import pandas as pd
import numpy as np

# CGGA

In [ ]:
cgga_path = "/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/CGGA/"

In [ ]:
clinical = pd.read_csv(cgga_path + "CGGA.mRNAseq_693_clinical.20200506.txt", sep="\t")
clinical

In [ ]:
clinic_gbms = clinical[clinical["Grade"]=="WHO IV"]
clinic_gbms

In [ ]:
clinic_primary = clinic_gbms[clinic_gbms["PRS_type"]=="Primary"] 

In [ ]:
clinic_primary

In [ ]:
samples = clinic_primary["CGGA_ID"].values

In [ ]:
filtered = clinic_primary[["CGGA_ID","OS","Censor (alive=0; dead=1)"]]

In [ ]:
filtered.set_index("CGGA_ID",inplace=True)

In [ ]:
expr = pd.read_csv(cgga_path + "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt", sep="\t", index_col=0)
expr

In [ ]:
expr = expr[samples]

In [ ]:
order = pd.read_csv("xval_001_order.csv",index_col=0).columns.values # read in the order from first execution for reproducability

In [ ]:
test = list(set.intersection(set(expr.index), set(order))) 

In [ ]:
expr_ordered = expr.reindex(order, fill_value=0)

In [ ]:
expr_ordered_t = expr_ordered.T

In [ ]:
expr_ordered_t

In [ ]:
merged = expr_ordered_t.join(filtered, how='inner')  # only keep overlapping samples

In [ ]:
merged["OS"] = merged["OS"]/365

In [ ]:
merged

In [ ]:
merged.rename(columns={'OS': 'OS_TIME', 'Censor (alive=0; dead=1)': 'OS_STATUS'}, inplace=True)

In [ ]:
merged

In [ ]:
merged.to_csv(cgga_path + "ccga_final_expression_with_survival.csv")